# tRIBS-Sandbox: Run Model

The goal for this notebook is to first run the model we made then use pytRIBS to load in our model outputs and make some plots from the model outputs. The plots shown below are just examples and each output file has multiple variables that can be plotted. Addtion detail on the available model outputs is located [Here](https://tribshms.readthedocs.io/en/latest/man/Output.html)

## Imports

In [ ]:
# Lets import the pytRIBS project and results classes that we need
from pytRIBS.classes import Project, Results

In [ ]:
# If you have installed pytRIBS, the following libraries should already be in your environment
import os, sys, shutil
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
from shapely.ops import unary_union
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
import pandas as pd

### Run The Model
In our previous notebook we got everything organized to run the model. Typically tRIBS is ran from the command line but in this codespace we setup a direct path to the tRIBS executeable so we run the executeable from within this notebook. 

So lets run the simulation:

In [ ]:
import time

# Define File Names
input_filename = 'SMF.in'  
log_filename   = 'simulation.log'

# Pre-run check
# Check if the input file actually exists before trying to run.
if not os.path.exists(input_filename):
    print(f"ERROR: Could not find input file: '{input_filename}'")
    print("Please check your spelling or ensure the previous pytRIBS step ran correctly.")
else:
    print(f"Found input file: {input_filename}")
    print(f"Starting tRIBS simulation...")
    print(f"Runtime output is being redirected to: {log_filename}")

    # Execute model
    # We use the 'time' module to track how long the simulation takes.
    start_time = time.time()
    
    # Run tRIBS! 
    # syntax: !tRIBS <input_file> > <log_file> 2>&1
    exit_code = os.system(f"tRIBS {input_filename} > {log_filename} 2>&1")
    
    end_time = time.time()
    duration = (end_time - start_time) / 60

    # Post-run report
    if exit_code == 0:
        print(f"\nSUCCESS: Simulation completed in {duration:.2f} minutes.")

## Results Class: Merge and Visualize Results
The Results class simplifies working with tRIBS outputs by offering post-processing methods that handle everything from file management to basic model output analysis. tRIBS generates a large amount of data with fine spatial and temporal resolutions, including time series of streamflow and spatially averaged state and flux variables. Additionally, the model produces Voronoi diagrams that can be used with both dynamic snapshots (captured at specific times) and integrated outputs (aggregated over the entire model run). The Results class helps manage these outputs, providing users with tools to merge parallel results and perform further analysis using commonly utilized data libraries.

In [ ]:
# Load results
name, epsg = 'SMF', 26912
proj = Project(os.getcwd(), name, epsg)
results = Results('SMF.in', meta=proj.meta)

A main benefit of the results class is the ability to point pytRIBS to your input file and from there it knows all of the file paths where the data is stored.

In [ ]:
gdf = results.voronoi.merge(results.int_spatial_vars,on='ID')
results.get_mrf_results()
results.get_element_results()

strmflw_sim_raw = results.get_qout_results()

### Visualize Results
First lets start with plotting the tRIBS outlet streamflow timeseries and compare it to the observations. Our ouputs and obervational data are not in the same time intervals so we need to do some preprocessing first.

In [ ]:
# We will use pytRIBS built-in ploting tools to make a plot of our observed and simulated streamflow for the modeled event
event_start='2014-08-12 16:00'
event_end='2014-08-12 23:00'

# Load observed data from the Excel file
obs_df = pd.read_excel('../init_data/met/SMF_Observations_1993-2025.xlsx',
                         sheet_name='Discharge', skiprows=6)
obs_df['datetime'] = pd.to_datetime(obs_df['Date'].astype(str) + ' ' + obs_df['Time'].astype(str))
obs_df.set_index('datetime', inplace=True)
observed = obs_df['cfs'] * 0.0283168          # convert units from CFS -> CMS

# Make Plot
results.plot_hydrograph(
    observed=observed,
    start=event_start,
    end=event_end,
    resample='5min',                          # align sim (3.75 min) and observations onto a common time step
    interpolate=True,                         # With this option will interpolate any missing values from the observations 
)
plt.show()

Now that we have organized and plotted the streamflow result we can look at some performance metrics.

In [ ]:

# Build the aligned (uniform) observed/simulated time series for the same window as the plot above
aligned = results.align_streamflow(observed, freq='10min',
                                   start=event_start, end=event_end)
obs = aligned['observed']
sim = aligned['simulated']
  
# PHYSICAL EVENT METRICS (Peaks, Timing, Volumes)
# Computed directly from the time-aligned data.
obs_peak,  sim_peak  = obs.max(), sim.max()
obs_tpeak, sim_tpeak = obs.idxmax(), sim.idxmax()

# Flow is in m^3/s; multiply by the (uniform) time step to get volume in m^3.
dt_seconds  = (aligned.index[1] - aligned.index[0]).total_seconds()
obs_vol_m3  = obs.sum() * dt_seconds
sim_vol_m3  = sim.sum() * dt_seconds
vol_error_pct = (sim_vol_m3 - obs_vol_m3) / obs_vol_m3 * 100


# STATISTICAL GOODNESS-OF-FIT Metrics
metrics = results.compute_metrics(obs, sim)   # {'NSE', 'KGE', 'PBIAS', 'RMSE'}

# Report
print("--- HYDROLOGICAL EVENT METRICS ---")
print(f"Observed Peak Flow: {obs_peak:.2f} m^3/s  (at {obs_tpeak.strftime('%m-%d %H:%M')})")
print(f"Simulated Peak Flow:{sim_peak:.2f} m^3/s  (at {sim_tpeak.strftime('%m-%d %H:%M')})")
print(f"Peak Timing Error:  {(sim_tpeak - obs_tpeak).total_seconds() / 3600:.1f} hours\n")

print(f"Observed Volume:    {obs_vol_m3:,.0f} m^3")
print(f"Simulated Volume:   {sim_vol_m3:,.0f} m^3")
print(f"Volume Error:       {vol_error_pct:+.1f}%\n")

print("--- STATISTICAL PERFORMANCE ---")
print(f"RMSE:               {metrics['RMSE']:.2f} m^3/s")
print(f"KGE:                {metrics['KGE']:.3f}")
print(f"Percent Bias:       {metrics['PBIAS']:+.1f}%")

While the metrics above aren't great remember that the first run is using the raw ADOT soil parameter values. Another important thing is that the streamflow gage is not necessarily suited for our exact purpose. The sensor itself is located approximately ~1ft above the ground surface, thus low flows are not recorded.

### Other Outputs, not used for direct calibration

tRIBS has many model outputs. One of those is spatial outputs. For this sandbox environment the model output the integrated spatial output file which has outputs like time invariant watershed properties but also cumulative outputs of flux variables like evapotranspiration.

Like with streamflow we can use the built-in pytRIBS plotting tools but here we will get a bit more advanced by using the tools to make a figure with mutliple subplots.

First spatial plot we will make is the voronoi polygon elevation map and the computational element ID:

In [ ]:
# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
results.plot_voronoi('Z', cmap='terrain', edgecolor='black', linewidth=0.2, ax=ax1,
                     legend_kwds={'label': 'Elevation (m)', 'shrink': 0.8})
results.plot_voronoi('ID', cmap='viridis', ax=ax2,
                     legend_kwds={'label': 'Polygon ID (Routing Order)', 'shrink': 0.8})
plt.tight_layout()
plt.show()

The Polygon ID map above is useful for understanding the model. You can see that the higher elevations cells around the boundary generally a lower Id value because these elements have no upstream element contributing flow to them i.e. they are computed first in the model. Vice versa with the channel elements they have higher Id values because they have many elements upstream of them.

As mentioned above, we can use the same output file to look at the cumulative evapotranspiration over the entire 480-hour simulation period.

In [ ]:
results.plot_voronoi('cET', cmap='YlOrBr',
                     legend_kwds={'label': 'Cumulative ET (mm)'})
plt.show()

Interestingly the model has some pretty large differences in cumulative ET. The plot of depth to bedrock was added as one way of getting to an explanation. Since the soils are so shallow on the mountainous slopes the vegetation in those areas has full access to any soil water available.

Another comonly used model output is the mean response file (`*.mrf`). This output contains the basin averaged timeseries of many of the model's important variables. 

Using another built-in function we can plot the mead depth to groundwater with the precipitation on the top axis to see how the model repsonds to rainfall.

In [ ]:
results.plot_mrf('MDGW_mm', invert=True)
plt.show()

Here we plotted to the groundwater depth from mrf file only the lower axis and mean areal precipitation on the top axis. We can see with the shallow depth to bedrock the soil is almost instantly saturated.

Now lets show an example for plotting the `*.pixel` file outputs. By default there were three random pixel IDs inputed selected to write outputs for in the Make_SMF_Model.ipynb notebook (you can change them yourself and rerun the modle if desired). Lets make a plot of one of those `*.pixel` files:

In [ ]:
results.get_element_results()                 # populates results.element
node_id = list(results.element.keys())[0]     # or set a specific node ID
results.plot_element(node_id, ['Rain_mm_h', 'Srf_Hour_mm', 'Nwt_mm'])
plt.show()

The plot above is a good example of one aspect of the model's internal soil-water physics. The bottom plot, `Nwt` is the depth to the water table. For this specific element the initial conditions was likely too wet by the fact that the water table started shallower then eventually stablized over a period of no rain but ET extracting the soil water. 

The large rainfall event occurs on the 12th but importantly the the depth to the water table does not change until the 17th. That might seem odd but the from the storm there is water in the upper column of the soil which is not shown with `Nwt`. The model has a parameter that controls (`INTERSTORMPERIOD`) that controls the time it takes for water to leave the upper soil layers into the water table.

Instead if you modify the code above to change the variable `MSMU_[]` to `SoilMoist_[]` you can see the how the volumetric soil mositure changes rapidly with the storm event. More variables that can be plotted are available on the wiki [here](https://tribshms.readthedocs.io/en/latest/man/Output.html) in section `6.1.2`. 